In [150]:
import lancedb
from sentence_transformers import SentenceTransformer
import json
import os
import pandas as pd
from openai import OpenAI
import re

In [177]:
DB_PATH = "/home/linux/linux/ProjectsReal/chef/scripts/lancedb"
TABLE_NAME = "recepies"

EMBED_MODEL = "all-MiniLM-L6-v2"
TOP_K = 10

In [63]:
# -----------------------------
# LOAD EMBEDDING MODEL
# -----------------------------
print("🔄 Loading embedding model...")
model = SentenceTransformer(EMBED_MODEL)


🔄 Loading embedding model...


Loading weights: 100%|█████████████████████████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 3443.38it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [64]:
def load_table():
	db = lancedb.connect(DB_PATH)
	return db.open_table(TABLE_NAME)

In [65]:
def parse_chatgpt_json(response: str):
    # Remove markdown code block if present
    cleaned = re.sub(r"```json|```", "", response).strip()
    
    return json.loads(cleaned)

In [184]:
# -----------------------------
# SEARCH FUNCTION
# -----------------------------
def search(table, query, top_k=5):
	# print(f"\n🔍 Query: {query}")

	query_embedding = model.encode(query).tolist()

	results = (
		table.search(query_embedding)
		.limit(top_k)
		.to_list()
	)

	return results

In [67]:
# -----------------------------
# DISPLAY RESULTS
# -----------------------------
def display_results(results):
    for i, r in enumerate(results):
        print(r)
        # print(r["text"])  # print first 500 chars
        print("\n")

In [185]:
def quertingLance(query):
    table = load_table()
    results = search(table, query, TOP_K)
    return results
	# display_results(results)

In [121]:
def truncate_response(query, answers_list):
    truncated_answer = []
    for answer in answers_list:
        answer_dict = {}
        answer_dict['query'] = query
        answer_dict['id'] = answer['id']
        answer_dict['text'] = answer['text']
        answer_dict['distance'] = answer['_distance']
        truncated_answer.append(answer_dict)
    return truncated_answer

In [189]:
# query = "breakfast with less than 5 minutes prep time?"
# query = "high protien low carbohydrates breakfast"
# query = "tomatoes and basil"
# query =  "dinner, quick, easy, low prep, fast"
query = "doemon shinchan pikcahu hayabusa"
query_response = quertingLance(query)
print(query_response)

[{'id': 381, 'recepie name': 'Nankhatais (Indian Cookies)', 'ingredients': ['150 gms flour', '100 gms ghee or clarified butter', '115 gms fine sugar', '1/8 tsp baking powder', '1/4 cup milk', '5 cardamoms, powdered', 'rose essence', 'blanched, chopped almonds or pistachios'], 'directions': ['Sift flour and baking powder.', 'Add sugar and mix well.', 'Work in ghee, cardamom powder and rose essence.', 'Add milk and knead to a soft dough.', 'Shape dough into small balls and press gently.', 'Garnish with chopped almonds or pistachios.', 'Bake in preheated moderate oven (180 degrees C) for 20 minutes or until golden brown.'], 'tags': ['Indian', 'Cookies', 'Baked', 'Sweet', 'Snack', 'Vegetarian', 'Dessert'], 'prep time': '15 minutes', 'cook time': '20 minutes', 'yield': 'About 20 cookies', 'cook time minutes': 20, 'prep time minutes': 20, 'per serving': 'Approximately 1 cookie', 'note': '', 'tip': '', 'vary it': 'For Mexican cookies, add half the quantity of sugar. Roll nankhatai dough into 

In [93]:
df = pd.read_csv('/home/linux/linux/ProjectsReal/chef/validation.csv')

In [94]:
df.head()

,Query,query type
0,Paneer curry recipe,ingredients
1,food items made with dal,ingredients
2,Indian Style Sandwich,ingredients
3,french fries recipes,ingredients
4,french onion soup tips,ingredients


In [122]:
full_results = []
for index, row in df.iterrows():
    query = row['Query']
    # print(query)
    query_response = quertingLance(query)
    # print(query_response)
    truncated = truncate_response(query, query_response)
    full_results = full_results + truncated

In [123]:
full_df = pd.DataFrame(full_results)

In [124]:
full_df.head()

,query,id,text,distance
0,Paneer curry recipe,309,Paneer Tikka ingredients = 500 gms home-made ...,0.582293
1,Paneer curry recipe,324,Garbanzos and Potato Curry ingredients = 1 ti...,0.836615
2,Paneer curry recipe,293,thai coconut curry ingredients = 150 gms long...,0.878080
3,Paneer curry recipe,343,Baked Curry Rice ingredients = 3 cups long gr...,0.887982
4,Paneer curry recipe,305,Yogurt Curry ingredients = 8-10 chick pea flo...,0.904629


In [125]:
full_df.to_csv("/home/linux/linux/ProjectsReal/chef/validation_extracted.csv", index=False)

# Validation

In [132]:
api_key = ""
client = OpenAI(api_key=api_key)

In [156]:
df = pd.read_csv('/home/linux/linux/ProjectsReal/chef/validation_extracted.csv')

In [157]:
df.head()

,query,text,distance,id
0,Paneer curry recipe,Paneer Tikka ingredients = 500 gms home-made ...,0.582293,309
1,Paneer curry recipe,Garbanzos and Potato Curry ingredients = 1 ti...,0.836615,324
2,Paneer curry recipe,thai coconut curry ingredients = 150 gms long...,0.878080,293
3,Paneer curry recipe,Baked Curry Rice ingredients = 3 cups long gr...,0.887982,343
4,Paneer curry recipe,Yogurt Curry ingredients = 8-10 chick pea flo...,0.904629,305


In [158]:
KNOWLEDGE_BASE = """
You are a recipe writer. you need to evaluate whether the recipe is related or not to the keyword and provide scores.
This is a part of a RAG system evaluation and I want to evaluate the system performance and accuracy.

Relevance:
Ensure the recipe is concise and relevant to the keyword
Give a score between 1 to 10 with 1 being least relevant and 10 being the most relevant.


Groundedness
Ensure the recipe is grounded in the keywords.
Ensure the recipe does not contain "hallucinated" information outside the scope of the keywords.
to what extent does the generated response agree with the retrieved context. 
give a score between 1 to 10 with 1 being least grounded and 10 being the most grounded.


relatedness:
You goal is to identify recipes which completely unrelated to the keyword
(2) If the recipe contains ANY words or semantic meaning related to the keyword, consider them related
It is OK if the recipe has SOME information that is unrelated to the keyword as long as (2) is met
give a score between 1 to 10 with 1 being least related and 10 being the most related.


semantic similarity:
rate semantic similarity between the keyword and the recipe with 10 being high semantic match and 1 being the lowest semantic match

Required Response:

{    
    "relevance"; score,
    "groundedness": score,
    "relatedness": score,
    "semantic match"; score
}

the score should strictly be as mentioned.
The response should strictly contain the keys and format as mentioned.
Please double check the response


    Return JSON.
"""

def extract_timeline(recipe, keyword):
    response = client.chat.completions.create(
        model="gpt-4.1-mini",  # fast + cheap + good for extraction
        temperature=0,
        messages=[
            {
                "role": "system",
                "content": KNOWLEDGE_BASE
            },
            {
                "role": "user",
                "content": f"Keyword:\n{keyword}  Recipe:\n{recipe}"
            }
        ]
    )

    result = response.choices[0].message.content
    return result
    # if result:
        # return json.loads(result)
    # return None

In [159]:
def parse_chatgpt_json(response: str):
    # Remove markdown code block if present
    cleaned = re.sub(r"```json|```", "", response).strip()
    return json.loads(cleaned)

In [160]:
keyw = "vegetarian soup recipes"
# recipetext = "Paneer Tikka ingredients =  500 gms home-made cottage cheese or paneer 1 thumb-size piece ginger 4 cloves garlic, minced 1 small onion, chopped Juice of one lemon 1/4 tsp black cumin seeds 1 whole cardamom, crushed 1/4 tsp powdered cloves 1/4 tsp powdered cinnamon 1/4 tsp chilli powder 2 tbsps tomato paste 1 cup yogurt 2 tbsps cream cheese Ghee or butter 1/4 cup fresh coriander, chopped directions =  Blend onion, ginger, garlic in blender. Blend further to a smooth paste with cream cheese and yogurt. Mix in all the ingredients. Cut cheese in cubes and marinate in yogurt mixture. Let stand for half an hour in a large saucepan. Heat on a very low flame for 15 minutes. Turn onto a serving platter, dot with butter and garnish with chopped coriander. Serve hot. tags =  main dish vegetarian Indian cuisine spicy dinner snack grilled appetizer prep time =  30 minutes cook time =  15 minutes yield =  Serves 4 variation =  For chilli mock duck: substitute mock duck for cheese cubes. Mock duck should be washed in hot water containing juice of half a lemon before cooking. Another variation is spicy beancurd: substitute deep fried cubes of soft beancurd for cheese cubes."
recipetext = "Tomato and Avocado Soup ingredients =  1 large or 2 small avocados 1 tin of tomatoes A good handful of fresh lovage Parsley to taste 1 stick of celery Small amount of sea salt 1 cup of apple juice 1 cup of water directions =  Place all the above in a blender and liquidise until smooth. Can be served chilled. tags =  cold dish soup raw vegetarian healthy nutritious gluten-free vegan prep time =  10 minutes cook time =  0 minutes yield =  4 servings per serving =  Highly nutritious, contains vitamins A, C and E"
scores = extract_timeline(recipetext, keyw)

In [161]:
print(scores)

{
    "relevance": 9,
    "groundedness": 10,
    "relatedness": 10,
    "semantic match": 9
}


In [151]:
df = df[:10]

In [162]:
scores_list = []
for index, row in df.iterrows():
    if index%20 == 0:
        print(index)
    query = row['query']
    text = row['text']
    distance = row['distance']
    id_ = row['id']
    scores = extract_timeline(text, query)
    scores = parse_chatgpt_json(scores)

    scores['query'] = query
    scores['text'] = text
    scores['distance'] = distance
    scores['id'] = id_
    scores_list.append(scores)
    

0
20
40
60
80
100
120
140
160
180
200
220
240
260
280
300
320
340
360
380
400
420
440
460
480
500
520
540
560
580
600
620
640


In [163]:
scores_df = pd.DataFrame(scores_list)

In [164]:
scores_df.head(20)

,relevance,groundedness,relatedness,semantic match,query,text,distance,id
0,4,5,6,4,Paneer curry recipe,Paneer Tikka ingredients = 500 gms home-made ...,0.582293,309
1,3,3,4,2,Paneer curry recipe,Garbanzos and Potato Curry ingredients = 1 ti...,0.836615,324
2,2,2,3,2,Paneer curry recipe,thai coconut curry ingredients = 150 gms long...,0.878080,293
3,4,5,6,4,Paneer curry recipe,Baked Curry Rice ingredients = 3 cups long gr...,0.887982,343
4,2,2,3,2,Paneer curry recipe,Yogurt Curry ingredients = 8-10 chick pea flo...,0.904629,305
5,9,10,10,9,food items made with dal,Lentil Bonda ingredients = 115 gms urad dal (...,0.959076,234
6,9,10,10,9,food items made with dal,Spicy Lentil Crumble ingredients = 450 gms gr...,0.976582,245
7,1,1,1,1,food items made with dal,Chinese Mustard Dressing ingredients = 1 tabl...,1.040651,87
8,1,1,1,1,food items made with dal,Rice Noodle Caboodle ingredients = water 1 ta...,1.057738,869
9,2,2,3,2,food items made with dal,"Udon noodles with seitan, broccoli, and napa c...",1.061356,67


In [165]:
scores_df.shape

(645, 8)

In [166]:
scores_df.columns

Index(['relevance', 'groundedness', 'relatedness', 'semantic match', 'query',
       'text', 'distance', 'id'],
      dtype='str')

In [167]:
scores_df = scores_df[['query', 'text', 'relevance', 'groundedness', 'relatedness', 'semantic match', 'distance', 'id']]

In [169]:
vali = pd.read_csv('/home/linux/linux/ProjectsReal/chef/validation.csv')

In [170]:
vali.head()

,Query,query type
0,Paneer curry recipe,ingredients
1,food items made with dal,ingredients
2,Indian Style Sandwich,ingredients
3,french fries recipes,ingredients
4,french onion soup tips,ingredients


In [171]:
scores_df_merged = scores_df.merge(vali, left_on="query", right_on="Query", how="left")

In [172]:
scores_df_merged.head()

,query,text,relevance,groundedness,relatedness,semantic match,distance,id,Query,query type
0,Paneer curry recipe,Paneer Tikka ingredients = 500 gms home-made ...,4,5,6,4,0.582293,309,Paneer curry recipe,ingredients
1,Paneer curry recipe,Garbanzos and Potato Curry ingredients = 1 ti...,3,3,4,2,0.836615,324,Paneer curry recipe,ingredients
2,Paneer curry recipe,thai coconut curry ingredients = 150 gms long...,2,2,3,2,0.878080,293,Paneer curry recipe,ingredients
3,Paneer curry recipe,Baked Curry Rice ingredients = 3 cups long gr...,4,5,6,4,0.887982,343,Paneer curry recipe,ingredients
4,Paneer curry recipe,Yogurt Curry ingredients = 8-10 chick pea flo...,2,2,3,2,0.904629,305,Paneer curry recipe,ingredients


In [173]:
scores_df_merged.drop(columns=["Query"], inplace=True)

In [174]:
scores_df_merged.head()

,query,text,relevance,groundedness,relatedness,semantic match,distance,id,query type
0,Paneer curry recipe,Paneer Tikka ingredients = 500 gms home-made ...,4,5,6,4,0.582293,309,ingredients
1,Paneer curry recipe,Garbanzos and Potato Curry ingredients = 1 ti...,3,3,4,2,0.836615,324,ingredients
2,Paneer curry recipe,thai coconut curry ingredients = 150 gms long...,2,2,3,2,0.878080,293,ingredients
3,Paneer curry recipe,Baked Curry Rice ingredients = 3 cups long gr...,4,5,6,4,0.887982,343,ingredients
4,Paneer curry recipe,Yogurt Curry ingredients = 8-10 chick pea flo...,2,2,3,2,0.904629,305,ingredients


In [175]:
scores_df_merged.to_csv('/home/linux/linux/ProjectsReal/chef/validation_extracted_scored_merged.csv', index=False)